# Price Cache Investigation
## Investigating anachronistic data and root causes

In [ ]:
import pandas as pd
import requests
import sys
sys.path.append('..')
import config
from pathlib import Path

## 1. Examine Cached Price Files
Let's look at the problematic tickers

In [ ]:
# Load a problematic ticker - RIVN (IPO 2021-11-10, but cache starts 1994)
rivn_df = pd.read_parquet('data/price/RIVN.parquet')

print("RIVN Price Data:")
print(f"Shape: {rivn_df.shape}")
print(f"Date range: {rivn_df.index.min()} to {rivn_df.index.max()}")
print(f"\nFirst 10 rows:")
rivn_df.head(10)

In [ ]:
# Last 10 rows (should be recent)
print("Last 10 rows:")
rivn_df.tail(10)

In [ ]:
# Check data around actual IPO date (2021-11-10)
ipo_window = rivn_df.loc['2021-11-01':'2021-11-20']
print("Data around IPO date (2021-11-10):")
ipo_window

In [ ]:
# Compare with another problematic ticker - SNOW (IPO 2020-09-16)
snow_df = pd.read_parquet('data/price/SNOW.parquet')

print("SNOW Price Data:")
print(f"Shape: {snow_df.shape}")
print(f"Date range: {snow_df.index.min()} to {snow_df.index.max()}")
print(f"\nFirst 10 rows:")
snow_df.head(10)

In [ ]:
# Check around SNOW's actual IPO (2020-09-16)
snow_ipo_window = snow_df.loc['2020-09-01':'2020-09-30']
print("SNOW data around IPO date (2020-09-16):")
snow_ipo_window

## 2. Attempt Fresh Download from FMP API
Let's see what FMP returns today for these tickers

In [ ]:
def test_fmp_download(ticker: str):
    """Test fresh download from FMP API."""
    url = f"{config.FMP_BASE_URL}/historical-price-eod/full"
    params = {
        'symbol': ticker,
        'from': '1990-01-01',
        'apikey': config.FMP_API_KEY
    }
    
    response = requests.get(url, params=params, timeout=30)
    
    print(f"\n{'='*60}")
    print(f"Testing {ticker}")
    print(f"{'='*60}")
    print(f"Status code: {response.status_code}")
    
    if response.status_code == 200:
        data = response.json()
        
        # Check response structure
        if isinstance(data, dict):
            print(f"Response type: dict")
            print(f"Keys: {data.keys()}")
            
            if 'symbol' in data:
                print(f"Symbol in response: {data['symbol']}")
            
            if 'historical' in data:
                historical = data['historical']
                print(f"Number of records: {len(historical)}")
                
                if len(historical) > 0:
                    # Convert to DataFrame
                    df = pd.DataFrame(historical)
                    df['date'] = pd.to_datetime(df['date'])
                    df = df.sort_values('date')
                    
                    print(f"\nDate range: {df['date'].min()} to {df['date'].max()}")
                    print(f"\nFirst 5 records:")
                    print(df.head())
                    
                    return df
        elif isinstance(data, list):
            print(f"Response type: list")
            print(f"Number of records: {len(data)}")
            
            if len(data) > 0:
                df = pd.DataFrame(data)
                if 'date' in df.columns:
                    df['date'] = pd.to_datetime(df['date'])
                    df = df.sort_values('date')
                    
                    print(f"Date range: {df['date'].min()} to {df['date'].max()}")
                    print(f"\nFirst 5 records:")
                    print(df.head())
                    
                    return df
    else:
        print(f"Error: {response.text[:200]}")
    
    return None

In [ ]:
# Test RIVN (should only have data from 2021-11-10 onwards)
rivn_fresh = test_fmp_download('RIVN')

In [ ]:
# Test SNOW (should only have data from 2020-09-16 onwards)
snow_fresh = test_fmp_download('SNOW')

In [ ]:
# Test RKLB (Rocket Lab - IPO 2021-08-25)
rklb_fresh = test_fmp_download('RKLB')

## 3. Check Company Profile API for IPO Dates
Let's see if FMP provides IPO dates in company profiles

In [ ]:
def get_company_profile(ticker: str):
    """Fetch company profile from FMP API."""
    url = f"{config.FMP_BASE_URL}/profile/{ticker}"
    params = {'apikey': config.FMP_API_KEY}
    
    response = requests.get(url, params=params, timeout=30)
    
    print(f"\n{'='*60}")
    print(f"Company Profile: {ticker}")
    print(f"{'='*60}")
    
    if response.status_code == 200:
        data = response.json()
        
        if isinstance(data, list) and len(data) > 0:
            profile = data[0]
            
            # Key fields
            print(f"Company Name: {profile.get('companyName', 'N/A')}")
            print(f"Symbol: {profile.get('symbol', 'N/A')}")
            print(f"IPO Date: {profile.get('ipoDate', 'N/A')}")
            print(f"Exchange: {profile.get('exchangeShortName', 'N/A')}")
            print(f"Industry: {profile.get('industry', 'N/A')}")
            print(f"\nFull profile keys available:")
            print(list(profile.keys()))
            
            return profile
        else:
            print(f"No profile data returned")
    else:
        print(f"Error: {response.status_code}")
    
    return None

In [ ]:
# Check profiles for problematic tickers
rivn_profile = get_company_profile('RIVN')

In [ ]:
snow_profile = get_company_profile('SNOW')

In [ ]:
rklb_profile = get_company_profile('RKLB')

In [ ]:
# Check a ticker that's NOT in the corrupted list (for comparison)
aapl_profile = get_company_profile('AAPL')

## 4. Compare Cached vs Fresh Download
Let's see if there are differences

In [ ]:
# Compare RIVN
if rivn_fresh is not None:
    print("RIVN Comparison:")
    print(f"\nCached data:")
    print(f"  Start date: {rivn_df.index.min()}")
    print(f"  End date: {rivn_df.index.max()}")
    print(f"  Rows: {len(rivn_df)}")
    
    print(f"\nFresh download:")
    print(f"  Start date: {rivn_fresh['date'].min()}")
    print(f"  End date: {rivn_fresh['date'].max()}")
    print(f"  Rows: {len(rivn_fresh)}")
    
    print(f"\n{'='*60}")
    print("ANALYSIS:")
    
    cached_start = pd.to_datetime(rivn_df.index.min())
    fresh_start = pd.to_datetime(rivn_fresh['date'].min())
    
    if cached_start < fresh_start:
        years_diff = (fresh_start - cached_start).days / 365.25
        print(f"CACHED DATA IS CORRUPTED!")
        print(f"  Cached data starts {years_diff:.1f} years earlier than FMP API returns today")
        print(f"  This confirms the cache was populated with wrong ticker data")
    else:
        print("Cached data aligns with FMP API")

## 5. Root Cause Analysis
Let's investigate WHEN this corruption might have occurred

In [ ]:
# Check file modification times
import os
from datetime import datetime

problematic_tickers = ['RIVN', 'SNOW', 'RKLB', 'RKT', 'RITM', 'U', 'LOAR']

print("File metadata for corrupted tickers:\n")
for ticker in problematic_tickers:
    file_path = Path(f'data/price/{ticker}.parquet')
    if file_path.exists():
        stat = file_path.stat()
        modified_time = datetime.fromtimestamp(stat.st_mtime)
        size_kb = stat.st_size / 1024
        
        print(f"{ticker}:")
        print(f"  Last modified: {modified_time}")
        print(f"  File size: {size_kb:.1f} KB")
        
        # Load and check
        df = pd.read_parquet(file_path)
        print(f"  Data range: {df.index.min()} to {df.index.max()}")
        print()

## 6. Check if yfinance was used vs FMP
Let's see what yfinance returns for these tickers

In [ ]:
import yfinance as yf

def test_yfinance_download(ticker: str):
    """Test what yfinance returns for a ticker."""
    print(f"\n{'='*60}")
    print(f"yfinance test: {ticker}")
    print(f"{'='*60}")
    
    try:
        data = yf.download(ticker, period='max', auto_adjust=True, progress=False)
        
        if not data.empty:
            print(f"Date range: {data.index.min()} to {data.index.max()}")
            print(f"Rows: {len(data)}")
            print(f"\nFirst 5 rows:")
            print(data.head())
            
            return data
        else:
            print("No data returned")
    except Exception as e:
        print(f"Error: {e}")
    
    return None

# Test RIVN with yfinance
rivn_yf = test_yfinance_download('RIVN')

In [ ]:
# Test SNOW with yfinance
snow_yf = test_yfinance_download('SNOW')

## 7. Hypothesis: Ticker Symbol Reuse
Could these ticker symbols have been used by different companies in the past?

In [ ]:
# Check if the old data prices match any known indices or other tickers
# This would confirm if wrong data was downloaded

# For RIVN - check the 1994 data
rivn_1994 = rivn_df.loc['1994-11-07':'1994-11-14']
print("RIVN cached data from 1994 (27 years before IPO):")
print(rivn_1994)

# These prices/patterns might match another ticker or index from that era

In [ ]:
# Compare multiple problematic tickers on the same date
# If they share data, they'll have identical values

check_date = '2003-04-02'  # Date from Dataset B duplicates

print(f"Checking price data on {check_date}:\n")

comparison_tickers = ['RITM', 'RIVN', 'RJF', 'RKLB', 'RKT', 'RL']

for ticker in comparison_tickers:
    file_path = Path(f'data/price/{ticker}.parquet')
    if file_path.exists():
        df = pd.read_parquet(file_path)
        
        if check_date in df.index:
            row = df.loc[check_date]
            print(f"{ticker}: Close={row['Close']:.6f}, Volume={row['Volume']:.0f}")
        else:
            print(f"{ticker}: No data on {check_date}")